# 1. فهم المشكلة

1. **العميل والمشكلة والتكلفة:** العميل هو إدارة شركة تجزئة تبيع الأثاث والتقنية والمستلزمات المكتبية. المشكلة أن الشركة تحقق مبيعات عالية (أكثر من 2.2 مليون دولار خلال 4 سنوات)، لكن تقريباً طلب من كل 5 طلبات (حوالي 20%) يسبب خسارة مالية.
2. **المتغير المستهدف (Target):** هو عمود `Profit` (ربح الطلب بالدولار)، ونوعه رقمي مستمر (Float).
3. **مستخدم المنتج وما يحتاجه:** سيستخدم المنتج مدير المبيعات أو مدير التسعير. أول ما يفتحه يحتاج أن يرى أداة بسيطة يختار فيها المنتج والمنطقة ونسبة الخصم، لتخبره مباشرة هل هذا الطلب غالباً رابح أم خاسر، وتوضح له أين تتركز الخسائر.
4. **3 أسئلة فرعية للإجابة عليها خلال الأسبوع:**
   - ما هي الفئة الفرعية من المنتجات (Sub-Category) الأكثر تسبباً في الخسائر؟
   - في أي مناطق أو مدن تتركز الطلبات الخاسرة؟
   - عند أي نسبة خصم (Discount) يبدأ الطلب بالتحول من ربح إلى خسارة؟

## 2. أول نظرة

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# قراءة البيانات
df = pd.read_csv('data/raw.csv', encoding='windows-1252')

# 1. معرفة الحجم (عدد الصفوف والأعمدة)
print("حجم البيانات:", df.shape)
df.head()

In [ ]:
# 2. معرفة أنواع البيانات والقيم الناقصة
df.info()
print("\nالقيم المفقودة في كل عمود:\n", df.isnull().sum())

In [ ]:
# 3. معرفة الصفوف المكررة والإحصائيات
print("عدد الصفوف المكررة:", df.duplicated().sum())
df.describe()

In [ ]:
# 4. أمثلة من القيم الفريدة في الأعمدة النصية المهمة
for col in ['Ship Mode', 'Segment', 'Region', 'Category']:
    print(f"--- {col} ---")
    print(df[col].unique())

## 3. التحليل الاستكشافي (EDA)

In [ ]:
# الرسمة 1: توزيع الـ Target (الربح)
plt.figure(figsize=(8, 4))
sns.histplot(df['Profit'], bins=50)
plt.title('Profit Distribution')
plt.xlim(-500, 500)
plt.show()

نلاحظ أن أغلب الطلبات تتركز أرباحها حول الصفر، وهناك عدد كبير من الطلبات تحقق خسارة تحت الصفر تصل إلى -6599.98 دولار.

In [ ]:
# الرسمة 2: العلاقة بين الخصم (Discount) والربح (Profit)
plt.figure(figsize=(8, 4))
sns.scatterplot(data=df, x='Discount', y='Profit')
plt.axhline(0, color='red', linestyle='--')
plt.title('Discount vs Profit')
plt.show()

من الرسمة يتضح أنه عندما يتجاوز الخصم نسبة 0.2 (20%)، تبدأ أغلب الطلبات بالتحول إلى خسارة تحت الخط الأحمر.

In [ ]:
# الرسمة 3: متوسط الربح حسب فئة المنتج (Category)
plt.figure(figsize=(8, 4))
sns.barplot(data=df, x='Category', y='Profit')
plt.title('Average Profit by Category')
plt.show()

تُظهر الرسمة أن فئة الأثاث (Furniture) هي الأقل ربحاً بمتوسط يبلغ حوالي 8.70 دولار، بينما تتفوق فئة التكنولوجيا بمتوسط 78.75 دولار.

# 4. مفاجآت الداتا

| المشكلة | العمود | كم صفاً؟ | قرارك المبدئي |
| :--- | :--- | :--- | :--- |
| وجود قيم سالبة (خسائر) تصل لأرقام كبيرة جداً | `Profit` | 1,871 صف | ليست خطأ، بل هي جوهر مشكلة العميل وسنحلل أسبابها غداً. |
| وجود خصومات عالية تصل إلى 80% (0.8) | `Discount` | 856 صف (أكثر من 50%) | طبيعية في البيانات لكنها السبب الرئيسي للخسارة، سنحدد حدها الآمن غداً. |
| أعمدة لا نحتاجها في التحليل مثل رقم الصف | `Row ID` | 9,994 صف | سنحذفها غداً أثناء تنظيف البيانات لأنها لا تفيد في التوقع. |
| نوع عمود التاريخ مخزن كنص (object) وليس تاريخ | `Order Date` | 9,994 صف | سنحوله غداً إلى صيغة تاريخ (datetime) لتسهيل التحليل الزمني. |